In [1]:
import scanpy as sc 
import pandas as pd 
import numpy as np 

import matplotlib.pyplot as plt 
import seaborn as sns 

from sklearn.metrics import adjusted_rand_score
from sklearn.decomposition import PCA

import scipy.sparse as sp 
import warnings

warnings.filterwarnings("ignore")

import os
import ctypes
import sys

# 1. 先设置 R_HOME
os.environ["R_HOME"] = "/home/pxy/miniconda3/envs/r40/lib/R"

# 2. 【核心黑科技】手动加载 R 的动态库
# 这步操作等同于在终端里设置 LD_LIBRARY_PATH，专门解决 VS Code 找不到库的问题
try:
    # 这是 R 的核心库路径
    libR_path = "/home/pxy/miniconda3/envs/r40/lib/R/lib/libR.so"
    # 强制加载进内存
    ctypes.CDLL(libR_path, mode=ctypes.RTLD_GLOBAL)
    print("✅ 成功强制加载 libR.so")
except OSError as e:
    print(f"❌ 加载失败: {e}")

# 3. 然后再导入其他包
sys.path.append("..") 

import spCLUE
import rpy2.robjects as robjects
print("R 环境路径:", robjects.r['R.home']()[0])

spCLUE.fix_seed(0)

# 定义DLPFC数据集的12个切片ID
slice_ids = [
    "151507", "151508", "151509", "151510",
    "151669", "151670", "151671", "151672",
    "151673", "151674", "151675", "151676"
]

# 用于存储每个切片的ARI结果
ari_results = []

# 数据路径（请根据实际情况确认路径是否正确）
data_dir = '/home/pxy/home/pxy/data/DLPFC/st/'

print(f"Start processing {len(slice_ids)} slices...")

for sample_name in slice_ids:
    print(f"\n{'='*20} Processing Sample: {sample_name} {'='*20}")
    
    # 1. 设置簇的数量 (根据DLPFC数据集的已知Ground Truth)
    # 151669-151672 通常只有5层，其他切片为7层
    if sample_name in ["151669", "151670", "151671", "151672"]:
        n_clusters = 5
    else:
        n_clusters = 7
    
    try:
        # 2. 加载数据
        # 使用 read_visium 加载数据，路径拼接逻辑参考原文件
        adata = sc.read_visium(data_dir + sample_name)
        adata.var_names_make_unique()
        
        # 加载元数据 (Ground Truth)
        meta = pd.read_csv(data_dir + sample_name + "/metadata.tsv", sep="\t")
        meta = meta.set_index("barcode")
        adata.obs["Region"] = meta.loc[adata.obs_names, "layer_guess_reordered"]
        
        # 3. 数据预处理与构图
        # 原文件 Cell 6 的逻辑
        adata = spCLUE.preprocess(adata, n_top_genes=3000)
        adata.obsm["X_pca"] = PCA(n_components=200, random_state=0).fit_transform(adata.X)
        
        g_spatial = spCLUE.prepare_graph(adata, "spatial", n_neighbors=6)
        g_expr = spCLUE.prepare_graph(adata, "expr", n_neighbors=8)
        graph_dict = {"spatial": g_spatial, "expr": g_expr}
        
        # 4. 模型初始化与训练
        # 原文件 Cell 8 的逻辑
        # 注意：这里将 n_clusters 参数改为动态变量，与当前切片保持一致
        spCLUE_model = spCLUE.spCLUE(
            input_data=adata.obsm['X_pca'],
            hvg_input=adata.obsm['X_pca'],
            graph_dict=graph_dict,
            n_clusters=n_clusters,
            reconstruction_loss='mse',
            gamma=1.0,
            gamma_mask=0.5,
            beta=1.0,
            kappa=5.0,
            # cluster_weight_strategy='min',  # 🔥 置信度策略
            # cluster_warmup_epochs=50,       # 🔥 Warmup
            # dim_input=2000,
            # dim_hidden=256,
            dim_embed=16,
            graph_corr=0.6,
            node_corr=0.3,
            epochs = 800,
            patience=50,
            min_delta=0.005,
        )
  
        # 训练模型
        _, adata.obsm["spCLUE"] = spCLUE_model.train()
        
        # 5. 聚类
        # 原文件 Cell 10 的逻辑
        refinement = True
        cluster_method = "mclust"
        spCLUE.clustering(
            adata,
            n_clusters,
            key="spCLUE",
            refinement=refinement,
            cluster_methods=cluster_method,
        )
        
        # 6. 计算 ARI
        # 原文件 Cell 12 的逻辑
        # 过滤掉 Ground Truth 为 NA 的区域
        adata_valid = adata[adata.obs.Region.notna()]
        ARI = adjusted_rand_score(adata_valid.obs["Region"], adata_valid.obs["mclust_refined"])
        
        print(f"Sample {sample_name} ARI: {ARI:.8f}")
        ari_results.append(ARI)
        
    except Exception as e:
        print(f"Error processing sample {sample_name}: {e}")

# 7. 输出最终统计结果
print(f"\n{'='*20} Final Results {'='*20}")
if ari_results:
    mean_ari = np.mean(ari_results)
    median_ari = np.median(ari_results)
    print(f"ARI per slice: {[round(x, 5) for x in ari_results]}")
    print(f"Mean ARI: {mean_ari:.4f}")
    print(f"Median ARI: {median_ari:.4f}")
else:
    print("No ARI results collected.")

✅ 成功强制加载 libR.so
R 环境路径: /home/pxy/miniconda3/envs/r40/lib/R
Start processing 12 slices...

==================== Processing Sample: 151507 ====================

预处理单切片数据 (全基因 PCA + 归一化 HVG)

根据原始计数挑选 top 3000 个高变基因...
全量数据归一化 (target_sum=1e4) & Log1p...
  ✓ 已保存归一化后的 HVG 重构目标: (4226, 3000)
将全量稀疏矩阵转换为稠密矩阵以计算 PCA...
计算 PCA (基于全基因归一化数据, n_components=200)...

✅ 预处理完成！
  - adata.X (全量归一化): (4226, 11982)
  - adata.obsm['X_hvg'] (归一化HVG): (4226, 3000)
  - adata.obsm['X_pca'] (全基因PCA): (4226, 200)

构建 spatial 图...
  - 使用空间坐标: (4226, 2)
  ✓ 空间图构建完成:  (4226, 4226), 边数=25832

构建 expr 图...
  - 使用PCA:  (4226, 200)
  ✓ 表达图构建完成: (4226, 4226), 边数=65578
ℹ️ 初始化单切片模型 (CCGCN)
   - 输入:  200维PCA
   - 重构: 2000个HVG
✅ 使用 MSE 重构损失
Training Start =========================>
重构损失类型: MSE
聚类对齐策略: min
Warmup epochs: 50
早停策略: patience=50, min_delta=0.005
损失权重: kappa=5.0, beta=1.0, gamma=1.0


  1%|          | 6/800 [00:01<02:11,  6.05it/s]


Epoch 1:   Loss=13.4793, ARI=0.129
 Cluster=2.5754, Recon=2.5867, MaskRecon=2.5876, LocalAgg=0.7023
  ClusterLoss=2.5629, NegLoss=0.0125


  7%|▋         | 55/800 [00:02<00:18, 39.78it/s]


Epoch 50:   Loss=12.0034, ARI=0.372
 Cluster=2.4119, Recon=2.5683, MaskRecon=2.5696, LocalAgg=0.5738
  ClusterLoss=2.4087, NegLoss=0.0033


...


Epoch 400:   Loss=8.6267, ARI=0.355
 Cluster=0.8537, Recon=2.4860, MaskRecon=2.4681, LocalAgg=0.4053
  ClusterLoss=0.8494, NegLoss=0.0043


 57%|█████▋    | 454/800 [00:10<00:08, 41.72it/s]


Epoch 450:   Loss=8.6887, ARI=0.360
 Cluster=0.8448, Recon=2.4862, MaskRecon=2.4809, LocalAgg=0.4117
  ClusterLoss=0.8401, NegLoss=0.0048


 57%|█████▋    | 459/800 [00:10<00:08, 41.97it/s]



Early stopping at epoch 460
Best loss: 7.8320
Training Finished =================<
fitting ...
  |======================================================================| 100%
Sample 151676 ARI: 0.39910616

==================== Final Results ====================
ARI per slice: [0.551, 0.44249, 0.43365, 0.54521, 0.37327, 0.30503, 0.70454, 0.56098, 0.46173, 0.41348, 0.37872, 0.39911]
Mean ARI: 0.4641
Median ARI: 0.4381
